# Example 1

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import matplotlib.pyplot as plt
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.io import VTXWriter


def exact_solution(x, D):
    return ufl.cos(ufl.pi * x[0]) * (1 - ufl.exp((x[1] - 1) / D)) / (
        1 - ufl.exp(-2 / D)
    ) + 0.5 * ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])


def solve_advection_diffusion(N, degree=1, D_value=0.1, w_mag=1.0):
    msh = mesh.create_unit_square(
        MPI.COMM_WORLD,
        N,
        N,
        cell_type=mesh.CellType.triangle,
        diagonal=mesh.DiagonalType.crossed,
    )

    print(type(msh))
    exit()

    V = fem.functionspace(msh, ("DG", degree))

    u = fem.Function(V)
    v = ufl.TestFunction(V)

    x = ufl.SpatialCoordinate(msh)
    n = ufl.FacetNormal(msh)
    h = ufl.CellDiameter(msh)

    D = fem.Constant(msh, PETSc.ScalarType(D_value))
    w = fem.Constant(msh, PETSc.ScalarType((0, w_mag)))

    u_exact = exact_solution(x, D)
    f = -ufl.div(D * ufl.grad(u_exact)) + ufl.dot(w, ufl.grad(u_exact))

    penalty = fem.Constant(msh, PETSc.ScalarType(10.0 * degree**2))

    dx = ufl.dx
    ds = ufl.ds
    dS = ufl.dS

    # Outflow indicator
    lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

    F = 0

    # Advection with upwind flux
    F += -ufl.inner(w * u, ufl.grad(v)) * dx
    F += ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v, n)) * dS
    F += ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds

    # Diffusion, symmetric interior penalty
    F += D * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx
    F += -D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v, n)) * dS
    F += -D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v))) * dS
    F += D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v, n)) * dS

    # Weak Dirichlet condition for diffusion on the whole boundary
    F += D * (
        -ufl.inner(ufl.grad(u), v * n) * ds
        - ufl.inner(ufl.grad(v), (u - u_exact) * n) * ds
        + (penalty / h) * ufl.inner(u - u_exact, v) * ds
    )

    # Inflow boundary condition for advection
    F += -ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_exact, v) * ds

    # Source
    F += -ufl.inner(f, v) * dx

    J = ufl.derivative(F, u)

    problem = NonlinearProblem(
        F,
        u,
        J=J,
        petsc_options_prefix="advecdiff",
        petsc_options={
            "snes_type": "newtonls",
            "snes_linesearch_type": "none",
            "snes_rtol": 1e-10,
            "snes_atol": 1e-10,
            "snes_max_it": 20,
            "ksp_type": "preonly",
            "pc_type": "lu",
        },
    )

    u = problem.solve()

    u.x.scatter_forward()

    # writer = VTXWriter(msh.comm, "DG_solution.bp", u, "BP5")
    # writer.write(t=0)

    error_form = fem.form((u - u_exact) ** 2 * dx)
    local_error = fem.assemble_scalar(error_form)
    l2_error = np.sqrt(msh.comm.allreduce(local_error, op=MPI.SUM))

    return l2_error


def convergence_test(degree=1):
    Ns = [8, 16, 32, 64, 128, 256]
    # Ns = [64]
    errors = []

    for N in Ns:
        error = solve_advection_diffusion(N, degree=degree, w_mag=50.0)
        errors.append(error)
        if MPI.COMM_WORLD.rank == 0:
            print(f"N = {N:3d}, L2 error = {error:.6e}")

    if MPI.COMM_WORLD.rank == 0:
        h = np.array([1 / N for N in Ns], dtype=float)
        errors = np.array(errors)

        plt.figure()
        plt.plot(h, errors)

        plt.ylabel("L2 error")
        plt.xlabel("Element size (h)")
        plt.xscale("log")
        plt.yscale("log")
        plt.grid(True, which="major", ls="--", lw=0.5)

        ax = plt.gca()
        ax.loglog(h, 2 * h**1.5, linestyle="--", color="black")
        ax.annotate(
            "Order 1.5",
            (h[0], 2 * h[0] ** 1.5),
            textcoords="offset points",
            xytext=(10, 0),
        )
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()


if __name__ == "__main__":
    convergence_test(degree=1)
